In [ ]:
import pandas as pd
train_df = pd.read_parquet(r"final_data\chunk-based-split-3\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"final_data\chunk-based-split-3\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"final_data\chunk-based-split-3\test_df_prepared.parquet", engine="pyarrow")


In [ ]:
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

In [ ]:
train_df = train_df.sort_values(by="timestamp_num").reset_index(drop=True)
valid_df = valid_df.sort_values(by="timestamp_num").reset_index(drop=True)
test_df = test_df.sort_values(by="timestamp_num").reset_index(drop=True)

In [ ]:
train_df = train_df.sort_values(by=['timestamp_num'])
valid_df = valid_df.sort_values(by=['timestamp_num'])
test_df = test_df.sort_values(by=['timestamp_num'])

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, GRU, Dropout
from tensorflow.keras.utils import to_categorical

# ---------------------------------------------------------
# 1. TÁCH DỮ LIỆU & ĐỊNH DẠNG ĐẦU VÀO
# ---------------------------------------------------------
print("Chuẩn bị dữ liệu...")
# Lấy X, y từ các tập dữ liệu
X_train = train_df.drop(columns=['label']).values
y_train = train_df['label'].values

X_valid = valid_df.drop(columns=['label']).values
y_valid = valid_df['label'].values

X_test = test_df.drop(columns=['label']).values
y_test = test_df['label'].values

num_classes = len(np.unique(y_train))
print(f"Số lượng classes: {num_classes}")

# One-hot encoding y cho các mô hình Neural Network
y_train_c = to_categorical(y_train, num_classes)
y_valid_c = to_categorical(y_valid, num_classes)
y_test_c = to_categorical(y_test, num_classes)

# Reshape X sang dạng 3D (samples, time_steps=1, features) cho LSTM/GRU
X_train_3d = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
X_valid_3d = X_valid.reshape(X_valid.shape[0], 1, X_valid.shape[1])
X_test_3d = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])


# ---------------------------------------------------------
# 2. XÂY DỰNG VÀ HUẤN LUYỆN CÁC MÔ HÌNH CƠ SỞ (TẦNG 1)
# ---------------------------------------------------------
print("\n--- Bắt đầu huấn luyện Tầng 1 (Level-1 Base Models) trên tập TRAIN ---")

# 1. Gradient Boosting
print("- Đang huấn luyện GBM...")
gbm = GradientBoostingClassifier()
gbm.fit(X_train, y_train)

# 2. LightGBM
print("- Đang huấn luyện LightGBM...")
lgbm = LGBMClassifier(verbose=-1)
lgbm.fit(X_train, y_train)

# 3. Bagging (1000 estimators)
print("- Đang huấn luyện Bagging (1000 cây)... (Có thể mất thời gian)")
bagging = BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=1000, n_jobs=-1)
bagging.fit(X_train, y_train)

# 4. XGBoost
print("- Đang huấn luyện XGBoost...")
xgb = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
xgb.fit(X_train, y_train)

# 5. CatBoost
print("- Đang huấn luyện CatBoost...")
cat = CatBoostClassifier(verbose=0)
cat.fit(X_train, y_train)

# 6. LSTM
print("- Đang huấn luyện LSTM...")
lstm_model = Sequential([
    LSTM(128, input_shape=(1, X_train.shape[1])),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])
lstm_model.compile(optimizer='adam', loss='categorical_crossentropy')
# Sử dụng số epoch nhỏ hoặc early stopping, ví dụ cụ thể để chạy
lstm_model.fit(X_train_3d, y_train_c, epochs=10, batch_size=1024, verbose=1)

# 7. GRU
print("- Đang huấn luyện GRU...")
gru_model = Sequential([
    GRU(128, input_shape=(1, X_train.shape[1])),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])
gru_model.compile(optimizer='adam', loss='categorical_crossentropy')
gru_model.fit(X_train_3d, y_train_c, epochs=10, batch_size=1024, verbose=1)


# ---------------------------------------------------------
# 3. TẠO CÁC ĐẶC TRƯNG META TỪ TẬP VALID (HOLD-OUT) VÀ TEST
# ---------------------------------------------------------
print("\n--- Trích xuất Meta-Features từ Tầng 1 ---")
def get_meta_features(X_2d, X_3d):
    p_gbm = gbm.predict_proba(X_2d)
    p_lgbm = lgbm.predict_proba(X_2d)
    p_bag = bagging.predict_proba(X_2d)
    p_xgb = xgb.predict_proba(X_2d)
    p_cat = cat.predict_proba(X_2d)
    p_lstm = lstm_model.predict(X_3d, verbose=0)
    p_gru = gru_model.predict(X_3d, verbose=0)
    
    # Nối theo chiều ngang (hstack) các prob predictions
    return np.hstack([p_gbm, p_lgbm, p_bag, p_xgb, p_cat, p_lstm, p_gru])

print("- Tạo dữ liệu huấn luyện Meta-Model trên tập VALID...")
X_meta_train = get_meta_features(X_valid, X_valid_3d)

print("- Tạo dữ liệu đánh giá Meta-Model trên tập TEST...")
X_meta_test = get_meta_features(X_test, X_test_3d)


# ---------------------------------------------------------
# 4. HUẤN LUYỆN META-MODEL (TẦNG 2) VÀ ĐÁNH GIÁ TRÊN TEST
# ---------------------------------------------------------
print("\n--- Huấn luyện Meta-Model (EnsembleGuard) Tầng 2 ---")
# Thiết kế Meta-Model: 1 lớp ẩn ẩn cơ bản theo yêu cầu
meta_model_nn = Sequential([
    Dense(64, activation='relu', input_shape=(X_meta_train.shape[1],)),
    Dense(num_classes, activation='softmax')
])

meta_model_nn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
# Huấn luyện trên tập Validation (X_meta_train, y_valid)
meta_model_nn.fit(X_meta_train, y_valid_c, epochs=20, batch_size=1024, verbose=1)


print("\n==============================================")
print("ĐÁNH GIÁ ENSEMBLE TRÊN TẬP TEST ĐỘC LẬP TỪ META-MODEL")
print("==============================================")

meta_preds_proba = meta_model_nn.predict(X_meta_test)
meta_preds_classes = np.argmax(meta_preds_proba, axis=1)

print("\n--- Báo cáo Phân loại (Classification Report) ---")
print(classification_report(y_test, meta_preds_classes, digits=4))

acc = accuracy_score(y_test, meta_preds_classes)
prec = precision_score(y_test, meta_preds_classes, average='macro', zero_division=0)
rec = recall_score(y_test, meta_preds_classes, average='macro', zero_division=0)
f1 = f1_score(y_test, meta_preds_classes, average='macro', zero_division=0)

print(f"\n=> Tỷ lệ chính xác (Accuracy): {acc:.4f}")
print(f"=> Macro Precision:          {prec:.4f}")
print(f"=> Macro Recall:             {rec:.4f}")
print(f"=> Macro F1-Score:           {f1:.4f}")